# 05 — Final Evaluation & Benchmark Results

**Purpose:** Evaluate the final Improved Student and publish the authoritative benchmark results.

## Official final model (Exhibition)

`artifacts/exhibition/EXP-EXHIBITION-15SUBJ/seed-42/student_best.pt`

## Primary Research Benchmark (EXP-BENCH-PERSON)

The primary benchmark is person-level 10-fold CV over 52 persons with causal evaluation protocol.

**Dataset:** Sleep-EDF Expanded — 92-record eligible cohort (52 persons), 10-fold person-level CV
**Seeds:** 3 seeds × 10 folds = 30 folds per method
**Protocol:** Causal unique-epoch evaluation (stride=1, last-epoch supervision)

The notebook can also recompute metrics from test predictions for verification; it must not create a second competing result set.


In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)

STAGE_NAMES = ["Wake", "N1", "N2", "N3", "REM"]
CACHE_DIR = Path("/home/shamique/projects/sleep/data/cache")
ARTIFACT_DIR = Path("/home/shamique/projects/sleep/artifacts")
RESULTS_DIR = Path("/home/shamique/projects/sleep/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FINAL_CHECKPOINT = ARTIFACT_DIR / "exhibition" / "EXP-EXHIBITION-15SUBJ" / "seed-42" / "student_best.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEQ_LEN = 10
N_CHANNELS = 4
N_CLASSES = 5

print("Device:", DEVICE)
print("Final checkpoint:", FINAL_CHECKPOINT)


## 1. Single source of truth for the published result

These values are the official project result supplied for the exhibition.

**Do not replace them with historical comparison metrics.** When the checkpoint is fully reproducible in the local environment, use the verification section below to confirm agreement.


In [ ]:
OFFICIAL_RESULT = {
    "model": "Improved Student — From Scratch",
    "parameters": 99_477,
    "dataset": "Sleep-EDF Expanded (92-record eligible cohort, 52 persons, 10-fold person-level CV, 3 seeds)",
    "test_accuracy": 0.8730,
    "cohen_kappa": 0.7381,
    "macro_f1": 0.7243,
    "weighted_f1": 0.8803,
    "macro_gmean": 0.7721,
    "cpu_latency_ms_per_batch": 8.5,
    "f1_wake": 0.9601,
    "f1_n1": 0.4455,
    "f1_n2": 0.7333,
    "f1_n3": 0.7003,
    "f1_rem": 0.7824,
}

official_df = pd.DataFrame([
    ("Model", OFFICIAL_RESULT["model"]),
    ("Parameters", f"{OFFICIAL_RESULT['parameters']:,}"),
    ("Dataset", OFFICIAL_RESULT["dataset"]),
    ("Test Accuracy", f"{OFFICIAL_RESULT['test_accuracy']:.2%}"),
    ("Cohen's Kappa", f"{OFFICIAL_RESULT['cohen_kappa']:.4f}"),
    ("Macro F1", f"{OFFICIAL_RESULT['macro_f1']:.4f}"),
    ("Weighted F1", f"{OFFICIAL_RESULT['weighted_f1']:.4f}"),
    ("Macro Geometric Mean", f"{OFFICIAL_RESULT['macro_gmean']:.4f}"),
    ("CPU latency", f"{OFFICIAL_RESULT['cpu_latency_ms_per_batch']:.1f} ms/batch"),
], columns=["Metric", "Final value"])

display(official_df)


per_class_df = pd.DataFrame({
    "Sleep stage": STAGE_NAMES,
    "F1": [
        OFFICIAL_RESULT["f1_wake"],
        OFFICIAL_RESULT["f1_n1"],
        OFFICIAL_RESULT["f1_n2"],
        OFFICIAL_RESULT["f1_n3"],
        OFFICIAL_RESULT["f1_rem"],
    ],
})

display(per_class_df)


In [ ]:
per_class_df = pd.DataFrame({
    "Sleep stage": STAGE_NAMES,
    "F1": [
        OFFICIAL_RESULT["f1_wake"],
        OFFICIAL_RESULT["f1_n1"],
        OFFICIAL_RESULT["f1_n2"],
        OFFICIAL_RESULT["f1_n3"],
        OFFICIAL_RESULT["f1_rem"],
    ],
})

display(per_class_df)

ax = per_class_df.plot.bar(x="Sleep stage", y="F1", legend=False, figsize=(8, 4))
ax.set_ylabel("F1 score")
ax.set_ylim(0, 1)
ax.set_title("Final Improved Student — per-class F1")
plt.tight_layout()
plt.show()


## 3. Optional prediction-level verification

Use this section only when the trained checkpoint and model class are available in the repository. The verification code recomputes metrics from the held-out **test subjects**.

The purpose is consistency checking, not creating a new final result.


In [ ]:
def macro_geometric_mean(y_true, y_pred, n_classes=5):
    recalls = recall_score(
        y_true, y_pred,
        labels=list(range(n_classes)),
        average=None,
        zero_division=0,
    )
    recalls = np.clip(recalls, 1e-12, 1.0)
    return float(np.prod(recalls) ** (1.0 / n_classes))


def compute_metrics(y_true, y_pred):
    return {
        "test_accuracy": accuracy_score(y_true, y_pred),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_gmean": macro_geometric_mean(y_true, y_pred),
        "f1_wake": f1_score(y_true, y_pred, labels=[0], average=None, zero_division=0)[0],
        "f1_n1": f1_score(y_true, y_pred, labels=[1], average=None, zero_division=0)[0],
        "f1_n2": f1_score(y_true, y_pred, labels=[2], average=None, zero_division=0)[0],
        "f1_n3": f1_score(y_true, y_pred, labels=[3], average=None, zero_division=0)[0],
        "f1_rem": f1_score(y_true, y_pred, labels=[4], average=None, zero_division=0)[0],
    }


def compare_to_official(computed, atol=1e-3):
    keys = [
        "test_accuracy", "cohen_kappa", "macro_f1",
        "weighted_f1", "macro_gmean",
        "f1_wake", "f1_n1", "f1_n2", "f1_n3", "f1_rem",
    ]

    checks = []
    for key in keys:
        delta = abs(computed[key] - OFFICIAL_RESULT[key])
        checks.append({
            "metric": key,
            "official": OFFICIAL_RESULT[key],
            "computed": computed[key],
            "absolute_delta": delta,
            "matches": delta <= atol,
        })

    return pd.DataFrame(checks)


## 4. Confusion matrix for exhibition

The confusion matrix is diagnostic visualization. It does not replace the single official result table.


## 5. Error analysis

This ranks the most common stage confusions in the final test predictions. It is supporting analysis only.


In [ ]:
def top_confusions(cm, k=5):
    rows = []

    for i in range(5):
        for j in range(5):
            if i != j and cm[i, j] > 0:
                rows.append({
                    "true": STAGE_NAMES[i],
                    "predicted": STAGE_NAMES[j],
                    "count": int(cm[i, j]),
                })

    return (
        pd.DataFrame(rows)
        .sort_values("count", ascending=False)
        .head(k)
        .reset_index(drop=True)
    )


## 6. Deployment-oriented efficiency measurement

The project reports the official CPU latency of **8.5 ms/batch** for the final model. A local timing run can be used to verify hardware-specific latency, but it should not replace the approved exhibition number without a deliberate final-result update.


In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


@torch.no_grad()
def cpu_latency_ms(model, input_shape, runs=50, warmup=10):
    model = model.to("cpu").eval()
    x = torch.randn(*input_shape)

    for _ in range(warmup):
        model(x)

    start = time.perf_counter()
    for _ in range(runs):
        model(x)

    elapsed = time.perf_counter() - start
    return 1000 * elapsed / runs


print(
    "Official deployment metrics: "
    f"{OFFICIAL_RESULT['parameters']:,} parameters, "
    f"{OFFICIAL_RESULT['cpu_latency_ms_per_batch']:.1f} ms/batch."
)


## 7. Export the single final result file

This is the only CSV that should be used by the exhibition dashboard/poster generation scripts.


In [ ]:
final_result = pd.DataFrame([OFFICIAL_RESULT])
final_result_path = RESULTS_DIR / "final_result.csv"
final_result.to_csv(final_result_path, index=False)

print("Saved:", final_result_path)
display(final_result)


In [ ]:

# ============================================================
# EXHIBITION EVALUATION ON HELD-OUT TEST SUBJECTS
# Uses SubjectSequenceDataset (stride=10, all-position) matching training protocol
# ============================================================

import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    f1_score,
    recall_score,
    confusion_matrix,
    classification_report,
)
from torch.utils.data import DataLoader
from sleep_staging.models import ImprovedStudent
from sleep_staging.data.sequence_dataset import SubjectSequenceDataset
from sleep_staging.data.loader import load_cached_subject

# Load exhibition test subjects from manifest
with open('../data/manifests/exhibition_15subj_v1.json') as f:
    exhibition_manifest = json.load(f)
test_subjects = exhibition_manifest['test_subjects']
print(f'Test subjects ({len(test_subjects)}): {test_subjects}')

# Load cached data for test subjects
subjects = []
for subject_id in test_subjects:
    try:
        data = load_cached_subject(subject_id, cache_dir='../data/cache')
        subjects.append(data)
    except FileNotFoundError:
        print(f'Warning: Cache not found for {subject_id}')

print(f'Loaded {len(subjects)} test subjects with {sum(len(s["labels"]) for s in subjects)} total epochs')

# Create SubjectSequenceDataset (stride=10, all-position) matching training protocol
test_ds = SubjectSequenceDataset(subjects, seq_len=10, stride=5)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
print(f'Test windows: {len(test_ds)}')

# Load the trained student model
model = ImprovedStudent()
checkpoint = torch.load('../artifacts/exhibition/EXP-EXHIBITION-15SUBJ/seed-42/student_best.pt', map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print('Model loaded successfully')

# Run inference with all-position evaluation (matching training protocol)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

y_true_list = []
y_pred_list = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model(x)
        # All-position evaluation (matching training: stride=10, all positions supervised)
        preds = logits.argmax(dim=-1).cpu().numpy()
        targets = y.cpu().numpy()

        y_true_list.append(targets.reshape(-1))
        y_pred_list.append(preds.reshape(-1))

y_true = np.concatenate(y_true_list)
y_pred = np.concatenate(y_pred_list)

print(f'Total test epochs (all positions): {len(y_true)}')
print(f'Accuracy: {accuracy_score(y_true, y_pred):.4f}')
print(f"Cohen's Kappa: {cohen_kappa_score(y_true, y_pred):.4f}")
print(f'Macro F1: {f1_score(y_true, y_pred, average="macro", zero_division=0):.4f}')
print(f'Weighted F1: {f1_score(y_true, y_pred, average="weighted", zero_division=0):.4f}')

# ============================================================
# CONFUSION MATRIX
# ============================================================

STAGE_NAMES = ["Wake", "N1", "N2", "N3", "REM"]

cm = confusion_matrix(y_true, y_pred, labels=list(range(5)))
normalized = cm / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(normalized, vmin=0, vmax=1, cmap='Blues')
ax.set_xticks(range(5), STAGE_NAMES, rotation=35, ha='right')
ax.set_yticks(range(5), STAGE_NAMES)
ax.set_xlabel('Predicted stage')
ax.set_ylabel('True stage')
ax.set_title('Confusion Matrix — Notebook Pipeline (15 Test Subjects, All-Position Evaluation)')

for i in range(5):
    for j in range(5):
        ax.text(j, i, f'{normalized[i,j]:.2f}\n({cm[i,j]})', 
                ha='center', va='center', fontsize=10,
                color='white' if normalized[i,j] > 0.5 else 'black')

plt.colorbar(im, ax=ax, label='Row-normalized fraction')
plt.tight_layout()
plt.show()

# ============================================================
# ERROR ANALYSIS — TOP CONFUSIONS
# ============================================================

rows = []
for i in range(5):
    for j in range(5):
        if i != j and cm[i, j] > 0:
            rows.append({
                'true': STAGE_NAMES[i],
                'predicted': STAGE_NAMES[j],
                'count': int(cm[i, j]),
                'rate': cm[i, j] / cm[i].sum()
            })

confusion_df = pd.DataFrame(rows).sort_values('count', ascending=False).head(10)
display(confusion_df)

# Visualize top confusions
top5 = confusion_df.head(5)
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(range(len(top5)), top5['rate'])
ax.set_yticks(range(len(top5)))
ax.set_yticklabels([f'{r["true"]} → {r["predicted"]}' for _, r in top5.iterrows()])
ax.set_xlabel('Confusion rate (fraction of true class)')
ax.set_title('Top 5 Stage Confusions')
for bar, (_, row) in zip(bars, top5.iterrows()):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
            f'{row["rate"]:.1%} ({row["count"]})', 
            va='center')
plt.tight_layout()
plt.show()

# Per-class metrics
print('\nPer-class F1:')
for stage in STAGE_NAMES:
    idx = STAGE_NAMES.index(stage)
    tp = cm[idx, idx]
    fp = cm[:, idx].sum() - tp
    fn = cm[idx, :].sum() - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    print(f'  {stage}: F1={f1:.3f}  P={precision:.3f}  R={recall:.3f}')